# Paso 3 -- Comparar las 4 recompensas + política base, sobre métricas reales

**Validación anti-Goodhart:** las 4 recompensas NO son comparables en
magnitud entre sí (A suma tiempo crudo, B/C/D son cuadráticas acotadas o
no -- ver Paso 1), así que esta comparación usa ÚNICAMENTE las métricas
reales del simulador (`metricas.reporte_completo`, reusado tal cual, sin
redefinir nada): tiempo medio y máximo en sistema, espera media,
movimientos, ocupación, y % atendidas (de referencia, no como criterio
principal -- mismo motivo que en `comparacion/README.md`: el simulador no
pierde a nadie, así que %atendidas depende en parte de dónde corta la
ventana, no solo de la política). Las 4 se evalúan, junto con la política
base, sobre las mismas 5 semillas de evaluación de siempre.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

BASE_DIR = Path.cwd().parent
SIMULADOR_DIR = (BASE_DIR / "../../simulador").resolve()
POLITICA_BASE_DIR = (BASE_DIR / "../../politica_base").resolve()
MODELO_RL_DIR = (BASE_DIR / "..").resolve()
BB_DIR = (BASE_DIR / "../../bergen-boats").resolve()
DEMAND_DIR = (BASE_DIR / "../../demand").resolve()
sys.path.insert(0, str(DEMAND_DIR / "src"))
sys.path.insert(0, str(SIMULADOR_DIR / "src"))
sys.path.insert(0, str(POLITICA_BASE_DIR / "src"))
sys.path.insert(0, str(MODELO_RL_DIR / "src"))
sys.path.insert(0, str(BASE_DIR / "src"))

import masas as demand_masas
import llegadas as demand_llegadas
from entrenamiento import EntornoDemandaAleatoria
from entorno_recompensa_intercambiable import EntornoRecompensaIntercambiable
from politica_base import asignar_flota
import metricas as met
import visualizacion as viz
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

cfg_propio = yaml.safe_load(open(BASE_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_sim = yaml.safe_load(open(SIMULADOR_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BB_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_demand = demand_masas.cargar_config(DEMAND_DIR / "config" / "instance.yaml")

resumen_masas = pd.read_csv(DEMAND_DIR / "output" / "masas_por_nodo.csv", index_col="id")
intensidad_od = pd.read_csv(DEMAND_DIR / "output" / "matriz_intensidad_od.csv")
poblacion_total_zonas = resumen_masas["poblacion_total"].sum()
conexiones_fuertes = cfg_bb["garantia"]["conexiones_fuertes"]
nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BB_DIR / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)

cfg_recompensas = cfg_propio["recompensas"]
cfg_entren_propio = cfg_propio["entrenamiento"]
cfg_agente = cfg_sim["agente"]
gamma = cfg_agente["gamma"]
cfg_escalon = cfg_sim["escalones"][cfg_entren_propio["escalon_base"]]
hora_ini_min, hora_fin_min = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60
semillas_eval = cfg_agente["evaluacion"]["semillas"]

OUT_MODELOS = BASE_DIR / "output" / "modelos"
OUT_COMPARACION = BASE_DIR / "output" / "comparacion"
OUT_COMPARACION.mkdir(parents=True, exist_ok=True)

print(f"Semillas de evaluacion: {semillas_eval}")


def construir_entorno(tipo_recompensa):
    return EntornoRecompensaIntercambiable(
        tipo_recompensa=tipo_recompensa,
        cfg_recompensa_tipo=cfg_recompensas[tipo_recompensa],
        gamma=gamma,
        generar_llegadas_dia_fn=demand_llegadas.generar_llegadas_dia,
        cfg_demand=cfg_demand, intensidad_od=intensidad_od, conexiones_fuertes=conexiones_fuertes,
        poblacion_total_zonas=poblacion_total_zonas, horas=cfg_escalon["horas"],
        porcentaje_poblacion_dia=cfg_escalon["porcentaje_poblacion_dia"],
        matriz_tiempos=matriz_tiempos, nodos=nodos, num_barcos=cfg_escalon["num_barcos"],
        capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"], nodo_inicial=cfg_bb["flota"]["nodo_inicial"],
        paso_tiempo_min=cfg_sim["paso_tiempo_min"], hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_sim["recompensa"], unidad_demanda=cfg_sim["unidad_demanda"],
    )


def construir_entorno_base():
    return EntornoDemandaAleatoria(
        generar_llegadas_dia_fn=demand_llegadas.generar_llegadas_dia,
        cfg_demand=cfg_demand, intensidad_od=intensidad_od, conexiones_fuertes=conexiones_fuertes,
        poblacion_total_zonas=poblacion_total_zonas, horas=cfg_escalon["horas"],
        porcentaje_poblacion_dia=cfg_escalon["porcentaje_poblacion_dia"],
        matriz_tiempos=matriz_tiempos, nodos=nodos, num_barcos=cfg_escalon["num_barcos"],
        capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"], nodo_inicial=cfg_bb["flota"]["nodo_inicial"],
        paso_tiempo_min=cfg_sim["paso_tiempo_min"], hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_sim["recompensa"], unidad_demanda=cfg_sim["unidad_demanda"],
    )


Semillas de evaluacion: [1001, 1002, 1003, 1004, 1005]


## Evaluar las 4 + base, sobre las 5 semillas

In [2]:
def evaluar_agente(tipo):
    out_dir_tipo = OUT_MODELOS / tipo
    model = PPO.load(str(out_dir_tipo / "modelo_ppo"))
    venv_stats = VecNormalize.load(str(out_dir_tipo / "vecnormalize.pkl"), DummyVecEnv([lambda: construir_entorno(tipo)]))
    venv_stats.training = False
    venv_stats.norm_reward = False

    envs = []
    for semilla in semillas_eval:
        env = construir_entorno(tipo)
        obs, info = env.reset(seed=semilla)
        while True:
            obs_norm = venv_stats.normalize_obs(obs)
            accion, _ = model.predict(obs_norm, deterministic=True)
            obs, r, terminated, truncated, info = env.step(accion)
            if truncated or terminated:
                break
        envs.append(env)
    return envs


def evaluar_base():
    envs = []
    for semilla in semillas_eval:
        env = construir_entorno_base()
        obs, info = env.reset(seed=semilla)
        while True:
            estado = info["_estado_obj"]
            libres = [b for b in estado.barcos if b.libre]
            decisiones = asignar_flota(libres, estado, matriz_tiempos, env.capacidad_barco, cfg_sim["recompensa"])
            accion = np.array([env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0 for b in estado.barcos])
            obs, r, terminated, truncated, info = env.step(accion)
            if truncated or terminated:
                break
        envs.append(env)
    return envs


corridas = {}
reportes = {}
for tipo in ["A", "B", "C", "D"]:
    print(f"Evaluando {tipo}...")
    envs = evaluar_agente(tipo)
    corridas[tipo] = met.combinar_corridas(envs)
    reportes[tipo] = met.reporte_completo(corridas[tipo])

print("Evaluando base...")
envs_base = evaluar_base()
corridas["base"] = met.combinar_corridas(envs_base)
reportes["base"] = met.reporte_completo(corridas["base"])

for tipo, rep in reportes.items():
    cuadra = rep["conservacion"]["cuadra"]
    print(f"[{tipo}] conservacion={cuadra}")


Evaluando A...
Evaluando B...
Evaluando C...
Evaluando D...
Evaluando base...
[A] conservacion=True
[B] conservacion=True
[C] conservacion=True
[D] conservacion=True
[base] conservacion=True


## Tabla de métricas reales (nunca el reward propio de cada una)

In [3]:
filas = []
for tipo in ["A", "B", "C", "D", "base"]:
    rep = reportes[tipo]
    filas.append({
        "tipo": tipo,
        "tiempo_sistema_medio_min": rep["globales"]["sistema_medio_min"],
        "tiempo_sistema_maximo_min": rep["globales"]["sistema_maximo_min"],
        "espera_media_min": rep["globales"]["espera_media_min"],
        "movimientos_totales": int(rep["por_barco"]["movimientos"].sum()),
        "ocupacion_media": float(rep["por_barco"]["ocupacion_media"].mean()),
        "pct_atendidas_referencia": rep["globales"]["pct_atendidas"],
    })

tabla_comparativa = pd.DataFrame(filas)
tabla_comparativa.to_csv(OUT_COMPARACION / "tabla_comparativa.csv", index=False)
display(tabla_comparativa.round(2))


,tipo,tiempo_sistema_medio_min,tiempo_sistema_maximo_min,espera_media_min,movimientos_totales,ocupacion_media,pct_atendidas_referencia
0,A,37.17,122.07,22.45,164,3.00,78.12
1,B,29.02,60.77,17.06,191,3.50,88.00
2,C,29.25,74.07,20.21,193,3.39,84.47
3,D,29.48,82.28,20.72,199,3.24,83.41
4,base,26.05,69.74,16.42,155,3.50,88.00


## ¿Cuál ganó?

Criterio (mismo que el resto del proyecto, ver `comparacion/README.md`):
menor tiempo en sistema (medio y, sobre todo, máximo) y menos movimientos
-- NO %atendidas. Se compara entre A/B/C/D (la política base es la
referencia externa, no compite por "ganar" acá).

In [4]:
solo_abcd = tabla_comparativa[tabla_comparativa["tipo"] != "base"].copy()
solo_abcd["rank_medio"] = solo_abcd["tiempo_sistema_medio_min"].rank()
solo_abcd["rank_maximo"] = solo_abcd["tiempo_sistema_maximo_min"].rank()
solo_abcd["rank_movimientos"] = solo_abcd["movimientos_totales"].rank()
solo_abcd["rank_total"] = solo_abcd["rank_medio"] + solo_abcd["rank_maximo"] + solo_abcd["rank_movimientos"]
display(solo_abcd[["tipo", "tiempo_sistema_medio_min", "tiempo_sistema_maximo_min", "movimientos_totales", "rank_total"]].sort_values("rank_total"))

ganador = solo_abcd.sort_values("rank_total").iloc[0]["tipo"]
print(f"\nGanador (menor suma de rangos en medio/maximo/movimientos): {ganador}")


,tipo,tiempo_sistema_medio_min,tiempo_sistema_maximo_min,movimientos_totales,rank_total
1,B,29.023593,60.77,191,4.0
2,C,29.248321,74.07,193,7.0
0,A,37.169692,122.07,164,9.0
3,D,29.479358,82.28,199,10.0



Ganador (menor suma de rangos en medio/maximo/movimientos): B


## Gráficas de comparación

In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

metricas_a_graficar = ["tiempo_sistema_medio_min", "tiempo_sistema_maximo_min", "espera_media_min", "movimientos_totales"]
fig = make_subplots(rows=2, cols=2, subplot_titles=metricas_a_graficar)
colores = {"A": "#2980b9", "B": "#27ae60", "C": "#8e44ad", "D": "#e67e22", "base": "#7f8c8d"}
for i, metrica in enumerate(metricas_a_graficar):
    row, col = i // 2 + 1, i % 2 + 1
    fig.add_trace(go.Bar(x=tabla_comparativa["tipo"], y=tabla_comparativa[metrica],
                          marker_color=[colores[t] for t in tabla_comparativa["tipo"]], showlegend=False),
                  row=row, col=col)
fig.update_layout(title="A / B / C / D / base -- metricas reales, 5 semillas", template="plotly_white", height=700)
fig.write_html(str(OUT_COMPARACION / "comparacion_metricas.html"))
fig.show()


## Detalle de la recompensa ganadora: desglose por par + perfil temporal

In [6]:
corrida_ganadora = corridas[ganador]

fig = viz.graficar_heatmap_cumplimiento(corrida_ganadora)
fig.write_html(str(OUT_COMPARACION / f"heatmap_por_par_{ganador}.html"))
fig.show()


In [7]:
fig = viz.graficar_perfil_espera(corrida_ganadora)
fig.write_html(str(OUT_COMPARACION / f"perfil_espera_{ganador}.html"))
fig.show()


In [8]:
fig = viz.graficar_ocupacion_flota(corrida_ganadora)
fig.write_html(str(OUT_COMPARACION / f"ocupacion_flota_{ganador}.html"))
fig.show()


## Nota: empate exacto en el ranking compuesto (C y D, 5 vs 5)

C y D quedan EXACTAMENTE empatados en la suma de rangos (5 cada una, contra 10 de A y B) --
el codigo de arriba elige uno por como pandas rompe empates internamente (no es una decision
metodologica real). Los dos representan compromisos distintos: D gana en tiempo medio y espera
media; C gana en el peor caso Y en movimientos (72, muy por debajo de las demas). Se muestra el
detalle de las dos, no solo la que el desempate automatico eligio -- ver seccion 6 del README
para la lectura completa.

In [9]:
corrida_C = corridas["C"]
fig = viz.graficar_heatmap_cumplimiento(corrida_C)
fig.write_html(str(OUT_COMPARACION / "heatmap_por_par_C.html"))
fig.show()


In [10]:
fig = viz.graficar_perfil_espera(corrida_C)
fig.write_html(str(OUT_COMPARACION / "perfil_espera_C.html"))
fig.show()


In [11]:
fig = viz.graficar_ocupacion_flota(corrida_C)
fig.write_html(str(OUT_COMPARACION / "ocupacion_flota_C.html"))
fig.show()
